In [ ]:
from sklearn.metrics import auc
from scipy.optimize import minimize
from sklearn.metrics import roc_curve
from sklearn.metrics import precision_recall_curve
from scipy.optimize import minimize
import joblib
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from sklearn.metrics import PrecisionRecallDisplay
from sklearn.metrics import RocCurveDisplay
from scipy import stats

In [ ]:
def getFromBucket(name_of_file_in_bucket):
    # get the bucket name
    my_bucket = os.getenv('WORKSPACE_BUCKET')

    # copy csv file from the bucket to the current working space
    os.system(f"gsutil cp '{my_bucket}/data/{name_of_file_in_bucket}' .")

    print(f'[INFO] {name_of_file_in_bucket} is successfully downloaded into your working space')
    # save dataframe in a csv file in the same workspace as the notebook
    my_dataframe = pd.read_csv(name_of_file_in_bucket)
    return my_dataframe.reset_index(drop=True)

In [ ]:
covid_features = getFromBucket('long_covid_features_2_25_2025.csv')

In [ ]:
survey_pred_vars = list(covid_features.loc[covid_features['survey_feature']==1,'feature_name'])

gene_pred_vars = list(covid_features.loc[covid_features['genetic_feature']==1,'feature_name'])

ehr_pred_vars = list(covid_features.loc[covid_features['ehr_feature']==1,'feature_name'])

mobile_device_pred_vars = list(covid_features.loc[covid_features['mobile_device_feature']==1,'feature_name'])

all_pred_vars = list(survey_pred_vars) + list(gene_pred_vars) + list(ehr_pred_vars)

In [ ]:
ensemble_results = []

for split_id in range(10):
    split_path = f"splits/split_{split_id}"
    test_df = pd.read_csv(os.path.join(split_path, "test.csv"))
    X_test_full = test_df.drop(columns=["target"])
    y_test = test_df["target"].values

    for fold_id in range(5):
        fold_path = os.path.join(split_path, f"fold_{fold_id}")
        val_df = pd.read_csv(os.path.join(fold_path, "val.csv"))
        X_val_full = val_df.drop(columns=["target"])
        y_val = val_df["target"].values

        # Load models
        model_A = joblib.load(f"xgboost_ehr_features_calibrated_split_{split_id}_fold_{fold_id}.pkl")
        model_B = joblib.load(f"xgboost_survey_features_calibrated_split_{split_id}_fold_{fold_id}.pkl")
        model_C = joblib.load(f"xgboost_gene_features_calibrated_split_{split_id}_fold_{fold_id}.pkl")

        # Select features for each model
        X_val_A = X_val_full[ehr_pred_vars]   # <- define A_features before loop
        X_val_B = X_val_full[survey_pred_vars]
        X_val_C = X_val_full[gene_pred_vars]

        X_test_A = X_test_full[ehr_pred_vars]
        X_test_B = X_test_full[survey_pred_vars]
        X_test_C = X_test_full[gene_pred_vars]

        # Predict probabilities
        probs_val_A = model_A.predict_proba(X_val_A)[:, 1]
        probs_val_B = model_B.predict_proba(X_val_B)[:, 1]
        probs_val_C = model_C.predict_proba(X_val_C)[:, 1]

        probs_test_A = model_A.predict_proba(X_test_A)[:, 1]
        probs_test_B = model_B.predict_proba(X_test_B)[:, 1]
        probs_test_C = model_C.predict_proba(X_test_C)[:, 1]

        # Stack
        probs_stack_val = np.vstack([probs_val_A, probs_val_B, probs_val_C]).T
        probs_stack_test = np.vstack([probs_test_A, probs_test_B, probs_test_C]).T

        # Optimize weights
        def weighted_ensemble_auc(weights, probs_stack, y_true):
            weights = np.array(weights)
            preds = np.dot(probs_stack, weights)
            prc = precision_recall_curve(y_true, preds)
            return -auc(prc[1], prc[0])

        initial_weights = [0.05, 0.9, 0.05]
        constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
        bounds = [(0, 1)] * 3

        result = minimize(weighted_ensemble_auc, initial_weights,
                          args=(probs_stack_val, y_val),
                          method='SLSQP',
                          bounds=bounds,
                          constraints=constraints)
        
#         print(result.success)

        optimal_weights = result.x
        ensemble_val_preds = np.dot(probs_stack_val, optimal_weights)
        ensemble_test_preds = np.dot(probs_stack_test, optimal_weights)

        prc_train = precision_recall_curve(y_val, ensemble_val_preds)
        val_auc = auc(prc_train[1], prc_train[0])
        
        prc_test = precision_recall_curve(y_test, ensemble_test_preds)
        test_auc = auc(prc_test[1], prc_test[0])

        ensemble_results.append({
            "split": split_id,
            "fold": fold_id,
            "val_auc": val_auc,
            "test_auc": test_auc,
            "w_A": optimal_weights[0],
            "w_B": optimal_weights[1],
            "w_C": optimal_weights[2]
        })

In [ ]:
ensemble_results_df = pd.DataFrame(ensemble_results)
ensemble_results_df.to_csv("ensemble_optimized_results.csv", index=False)

print(ensemble_results_df[["val_auc", "test_auc"]].describe())

In [ ]:
print("Sum of weights:", np.sum(result.x))
print(ensemble_results_df[["w_A", "w_B", "w_C"]].describe())

minimize() method with log loss

In [ ]:
from sklearn.metrics import log_loss

ensemble_results = []

for split_id in range(10):
    split_path = f"splits/split_{split_id}"
    test_df = pd.read_csv(os.path.join(split_path, "test.csv"))
    X_test_full = test_df.drop(columns=["target"])
    y_test = test_df["target"].values

    for fold_id in range(5):
        fold_path = os.path.join(split_path, f"fold_{fold_id}")
        val_df = pd.read_csv(os.path.join(fold_path, "val.csv"))
        X_val_full = val_df.drop(columns=["target"])
        y_val = val_df["target"].values

        # Load models
        model_A = joblib.load(f"xgboost_ehr_features_calibrated_split_{split_id}_fold_{fold_id}.pkl")
        model_B = joblib.load(f"xgboost_survey_features_calibrated_split_{split_id}_fold_{fold_id}.pkl")
        model_C = joblib.load(f"xgboost_gene_features_calibrated_split_{split_id}_fold_{fold_id}.pkl")

        # Select features for each model
        X_val_A = X_val_full[ehr_pred_vars]   # <- define A_features before loop
        X_val_B = X_val_full[survey_pred_vars]
        X_val_C = X_val_full[gene_pred_vars]

        X_test_A = X_test_full[ehr_pred_vars]
        X_test_B = X_test_full[survey_pred_vars]
        X_test_C = X_test_full[gene_pred_vars]

        # Predict probabilities
        probs_val_A = model_A.predict_proba(X_val_A)[:, 1]
        probs_val_B = model_B.predict_proba(X_val_B)[:, 1]
        probs_val_C = model_C.predict_proba(X_val_C)[:, 1]

        probs_test_A = model_A.predict_proba(X_test_A)[:, 1]
        probs_test_B = model_B.predict_proba(X_test_B)[:, 1]
        probs_test_C = model_C.predict_proba(X_test_C)[:, 1]

        # Stack
        probs_stack_val = np.vstack([probs_val_A, probs_val_B, probs_val_C]).T
        probs_stack_test = np.vstack([probs_test_A, probs_test_B, probs_test_C]).T

        # Optimize weights
        def weighted_ensemble_logloss(weights, probs_stack, y_true):
            weights = np.array(weights)
            preds = np.dot(probs_stack, weights)
            preds = np.clip(preds, 1e-6, 1 - 1e-6)
            return log_loss(y_true, preds)

        initial_weights = [1/3, 1/3, 1/3]
        constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
        bounds = [(0, 1)] * 3

        result = minimize(weighted_ensemble_logloss, initial_weights,
                          args=(probs_stack_val, y_val),
                          method='SLSQP',
                          bounds=bounds,
                          constraints=constraints)
        
#         print(result.success)

        optimal_weights = result.x
        ensemble_val_preds = np.dot(probs_stack_val, optimal_weights)
        ensemble_test_preds = np.dot(probs_stack_test, optimal_weights)

        prc_train = precision_recall_curve(y_val, ensemble_val_preds)
        val_auc = auc(prc_train[1], prc_train[0])
        
        prc_test = precision_recall_curve(y_test, ensemble_test_preds)
        test_auc = auc(prc_test[1], prc_test[0])

        ensemble_results.append({
            "split": split_id,
            "fold": fold_id,
            "val_auc": val_auc,
            "test_auc": test_auc,
            "w_A": optimal_weights[0],
            "w_B": optimal_weights[1],
            "w_C": optimal_weights[2]
        })

In [ ]:
ensemble_results_df = pd.DataFrame(ensemble_results)
ensemble_results_df.to_csv("ensemble_optimized_logloss_results.csv", index=False)

print(ensemble_results_df[["val_auc", "test_auc"]].describe())
print("Sum of weights:", np.sum(result.x))
print(ensemble_results_df[["w_A", "w_B", "w_C"]].describe())
print('\n')

# Extract AUROC values
val_aucs = ensemble_results_df["val_auc"].values
test_aucs = ensemble_results_df["test_auc"].values

# Sample size
n = len(val_aucs)  # should be 50

# Compute means and standard errors
val_mean = np.mean(val_aucs)
val_std = np.std(val_aucs, ddof=1)  # sample std

test_mean = np.mean(test_aucs)
test_std = np.std(test_aucs, ddof=1)

val_ci = stats.t.interval(confidence=0.95, df=n-1, 
              loc=val_mean, 
              scale=val_std/np.sqrt(n)) 
test_ci = stats.t.interval(confidence=0.95, df=n-1, 
              loc=test_mean, 
              scale=test_std/np.sqrt(n)) 

print(f"val_auc  mean: {val_mean:.4f}, 95% CI: ({val_ci[0]:.4f}, {val_ci[1]:.4f})")
print(f"test_auc mean: {test_mean:.4f}, 95% CI: ({test_ci[0]:.4f}, {test_ci[1]:.4f})")

gridsearch

In [ ]:
import itertools

def grid_search_weights(probs_stack, y_true, step=0.05):
    best_auc = -np.inf
    best_weights = None
    for w1, w2 in itertools.product(np.arange(0, 1 + step, step), repeat=2):
        if w1 + w2 <= 1.0:
            w3 = 1.0 - w1 - w2
            weights = np.array([w1, w2, w3])
            preds = np.dot(probs_stack, weights)
            try:
                #auc = roc_auc_score(y_true, preds)
                prc_train = precision_recall_curve(y_true, preds)
                auc = auc(prc_train[1], prc_train[0])
                if auc > best_auc:
                    best_auc = auc
                    best_weights = weights
            except Exception:
                continue
    return best_weights, best_auc

In [ ]:
ensemble_results = []

for split_id in range(10):
    split_path = f"splits/split_{split_id}"
    test_df = pd.read_csv(os.path.join(split_path, "test.csv"))
    X_test_full = test_df.drop(columns=["target"])
    y_test = test_df["target"].values
    
    print(split_id)

    for fold_id in range(5):
        fold_path = os.path.join(split_path, f"fold_{fold_id}")
        val_df = pd.read_csv(os.path.join(fold_path, "val.csv"))
        X_val_full = val_df.drop(columns=["target"])
        y_val = val_df["target"].values

        # Load models
        model_A = joblib.load(f"xgboost_ehr_features_calibrated_split_{split_id}_fold_{fold_id}.pkl")
        model_B = joblib.load(f"xgboost_survey_features_calibrated_split_{split_id}_fold_{fold_id}.pkl")
        model_C = joblib.load(f"xgboost_gene_features_calibrated_split_{split_id}_fold_{fold_id}.pkl")

        # Select features for each model
        X_val_A = X_val_full[ehr_pred_vars]   # <- define A_features before loop
        X_val_B = X_val_full[survey_pred_vars]
        X_val_C = X_val_full[gene_pred_vars]

        X_test_A = X_test_full[ehr_pred_vars]
        X_test_B = X_test_full[survey_pred_vars]
        X_test_C = X_test_full[gene_pred_vars]

        # Predict probabilities
        probs_val_A = model_A.predict_proba(X_val_A)[:, 1]
        probs_val_B = model_B.predict_proba(X_val_B)[:, 1]
        probs_val_C = model_C.predict_proba(X_val_C)[:, 1]

        probs_test_A = model_A.predict_proba(X_test_A)[:, 1]
        probs_test_B = model_B.predict_proba(X_test_B)[:, 1]
        probs_test_C = model_C.predict_proba(X_test_C)[:, 1]

        # Stack
        probs_stack_val = np.vstack([probs_val_A, probs_val_B, probs_val_C]).T
        probs_stack_test = np.vstack([probs_test_A, probs_test_B, probs_test_C]).T

        optimal_weights, val_auc = grid_search_weights(probs_stack_val, y_val)

        ensemble_val_preds = np.dot(probs_stack_val, optimal_weights)
        ensemble_test_preds = np.dot(probs_stack_test, optimal_weights)
        
        prc_test = precision_recall_curve(y_test, ensemble_test_preds)
        test_auc = auc(prc_test[1], prc_test[0])
        #test_auc = roc_auc_score(y_test, ensemble_test_preds)

        ensemble_results.append({
            "split": split_id,
            "fold": fold_id,
            "val_auc": val_auc,
            "test_auc": test_auc,
            "w_A": optimal_weights[0],
            "w_B": optimal_weights[1],
            "w_C": optimal_weights[2]
        })
        
        print(fold_id)
        
    print('#'*20)

In [ ]:
ensemble_results_df = pd.DataFrame(ensemble_results)
ensemble_results_df.to_csv("ensemble_optimized_gridsearch_results.csv", index=False)

print(ensemble_results_df[["val_auc", "test_auc"]].describe())
print("Sum of weights:", np.sum(result.x))
print(ensemble_results_df[["w_A", "w_B", "w_C"]].describe())
print('\n')

# Extract AUROC values
val_aucs = ensemble_results_df["val_auc"].values
test_aucs = ensemble_results_df["test_auc"].values

# Sample size
n = len(val_aucs)  # should be 50

# Compute means and standard errors
val_mean = np.mean(val_aucs)
val_std = np.std(val_aucs, ddof=1)  # sample std

test_mean = np.mean(test_aucs)
test_std = np.std(test_aucs, ddof=1)

val_ci = stats.t.interval(confidence=0.95, df=n-1, 
              loc=val_mean, 
              scale=val_std/np.sqrt(n)) 
test_ci = stats.t.interval(confidence=0.95, df=n-1, 
              loc=test_mean, 
              scale=test_std/np.sqrt(n)) 

print(f"val_auc  mean: {val_mean:.4f}, 95% CI: ({val_ci[0]:.4f}, {val_ci[1]:.4f})")
print(f"test_auc mean: {test_mean:.4f}, 95% CI: ({test_ci[0]:.4f}, {test_ci[1]:.4f})")